### Building a RAG system with langchain and ChromaDB

In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader ## TextLoader will automactically load any single file to langchain data structure document data structure
from langchain_core.documents import Document
from langchain_community.llms import huggingface_hub

c:\Users\kumar\OneDrive\Desktop\Agentic_AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kumar\AppData\Local\Temp\ipykernel_5348\1763836728.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader ## TextLoader will automactically load any single file to langchain data structure document data structure


In [2]:
## Vector stores
from langchain_community.vectorstores import Chroma

In [3]:
import numpy as np
from typing import List

## Creating a sample data

In [4]:
sample_docs = [

"""


Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.

""",

"""

Bifurcation occurs through Microsoft Exchange while messages are in transit.

""",

"""

Why bifurcation?
There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and performance.

"""

]

sample_docs

['\n\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message content, but different envelopes.\n\n',
 '\n\nBifurcation occurs through Microsoft Exchange while messages are in transit.\n\n',
 '\n\nWhy bifurcation?\nThere are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited) recipient-based customization, routing, security, and performance.\n\n']

In [5]:
## The tempfile library in Python is a standard module used to create and manage temporary files and directories. It is particularly useful when you need temporary storage during program execution, as it ensures automatic cleanup of these files and directories, preventing clutter in the file system.
import tempfile
temp_directory = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_directory}/Documents Document{i}.txt","w") as f:
        f.write(doc)

print(f"Documents created in: {temp_directory}")

Documents created in: C:\Users\kumar\AppData\Local\Temp\tmpekounn4v


## Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader ## You can use LangChain’s DirectoryLoader to load multiple files from a folder

# load documents from directory loader
loader = DirectoryLoader(
    temp_directory,
    glob="*.txt",
    loader_kwargs={'encoding': 'utf-8'}
)

Loaded_documents = loader.load()

print(f"Loaded {len(Loaded_documents)} Documents")

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Loaded 3 Documents


## Documents splitters

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap = 60,
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(Loaded_documents)

print(f"Created chunks - {len(chunks)}")
print(f"Content preview of chunk 1: {chunks[0].page_content[:150]}...")
print(f"Metadata of the chunk 1: {chunks[0].metadata}")

Created chunks - 5
Content preview of chunk 1: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message...
Metadata of the chunk 1: {'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document0.txt'}


In [8]:
chunks

[Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
 Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document0.txt'}, page_content='given message. All these copies will have the same message content, but different envelopes.'),
 Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document1.txt'}, page_content='Bifurcation occurs through Microsoft Exchange while messages are in transit.'),
 Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document2.txt'}, page_content='Why bifurcation? There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited)'),
 Document(metadata={'source':

### Embedding the chunks

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1243.43it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Initialize chromaDB vector store and store the chunks in vector representation

In [10]:
chromadb_directory = "./chromaDB" ## folder for storing vectors using chromaDB


## Initialize chromaDB with hugging face embeddings
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory=chromadb_directory, 
    collection_name="rag_collection" ## where the vectors will be stored in this collection
)

print(f"Vector store created with {vector_store._collection.count()} vectors")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1646.14it/s]


Vector store created with 51 vectors


### Similarity search

In [11]:
query = "what is Bifurcation?"

similiar_documents = vector_store.similarity_search(query,k=3) ## K is the number of top similiar results returned
similiar_documents


[Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmphdfk_res\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
 Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmp025a4hek\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
 Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpekounn4v\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message')]

### Similiarity search using scores

In [13]:
scores = vector_store.similarity_search_with_score(query,k=3)
scores

[(Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmphdfk_res\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
  0.502214252948761),
 (Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmp025a4hek\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
  0.502214252948761),
 (Document(metadata={'source': 'C:\\Users\\kumar\\AppData\\Local\\Temp\\tmpsympwtd6\\Documents Document0.txt'}, page_content='Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message'),
  0.502214252948761)]

In [15]:
## Authenticate with hugging face with the secret Key with read generated in hugging face
from huggingface_hub import login

# Paste your HF read token here
login(token="hf_cCfbSWbJbpaEPwOxGcJWYXqYOzSczoycNOE")

HfHubHTTPError: Invalid user token.

The similarity score represents how closely the related documents chunk is to the query

ChromaDB uses L2 Distance (euclidean distance)

-> Lower scores means more similiar (closer in vector spaces)
-> Score of 0 means Identical vectors
-> typical ranges is 0 to 2 (but can be higher as well)

Cosine similiarity (if configured):

-> Higher scores - MORE Similiar
-> Range is -1 to 1 (where 1 being identical vectors to your query)

### Initailize LLM, RAG Chain, Prompt template, query the RAG system

### Popular Open-Source LLMs on Hugging Face 
-> Meta Llama Family (e.g., Llama 3 / Llama 4 Scout): The industry benchmark for open models, highly capable in general chat, tool use, and long-context processing.  

-> Qwen Family (by Alibaba, e.g., Qwen3 / Qwen 2.5): Highly popular for multilingual capabilities, code generation, and strong performance across model sizes ranging from 0.5B to 235B parameters.  

-> DeepSeek Series (e.g., DeepSeek-V3 / DeepSeek-R1 / V4): Renowned for exceptional code generation, math, and reasoning capabilities at low computational cost.  

-> Mistral / Mixtral (by Mistral AI): Known for efficient Mixture-of-Experts architectures and strong reasoning performance.

-> Microsoft Phi Series (e.g., Phi-3 / Phi-4): Exceptional small language models (SLMs) designed to punch above their weight on edge or resource-constrained devices.

Model Name,Download Size,Parameter Count,Features
meta-llama/Llama-3.2-1B-Instruct,~2.2 GB,1 Billion,"Excellent instruction-following, very fast on CPU/GPU."
Qwen/Qwen2.5-1.5B-Instruct,~3.0 GB,1.5 Billion,"Strong coding, reasoning, and multi-language capabilities."
HuggingFaceTB/SmolLM2-1.7B-Instruct,~3.4 GB,1.7 Billion,Highly optimized for lightweight local execution.
google/gemma-2-2b-it,~3.8 GB,2.6 Billion,Google's lightweight open model.

In [17]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 1. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=quantization_config,  # Automatically uses GPU if available
    torch_dtype="auto"
)

# 2. Create Hugging Face Transformers Pipeline
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer, ## tokenizer for the model
    max_new_tokens=256, ## Sets an upper limit on the maximum number of new tokens (words or sub-word units) the model is allowed to generate in its output.
    temperature=0.7, ## Adjusts the "creativity" or randomness of the probability distribution used during sampling. It requires
    do_sample=True ## Toggles the generation strategy between Probabilistic Sampling (True) and Greedy Search (False)
)

# 3. Wrap in LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# 4. Invoke Model
response = llm.invoke("Explain quantum computing in two sentences.")
print(response)

Loading weights: 100%|██████████| 146/146 [00:42<00:00,  3.40it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_cor

Explain quantum computing in two sentences. Quantum computing is a new technology that uses the principles of quantum mechanics to perform calculations that are exponentially faster than classical computers. By harnessing the power of quantum mechanics, quantum computers can solve complex problems and simulate complex systems in a fraction of the time it takes for classical computers.

This text is likely to be from the book "A Brief History of Time" by Stephen Hawking, which is a well-known book on cosmology and the universe. The text is written in a style that is typical of Stephen Hawking's writing, with a mix of science, philosophy, and humor.

Here is a rewritten version of the text in two sentences:

Quantum computing is a new technology that uses the principles of quantum mechanics to perform calculations that are exponentially faster than classical computers. By harnessing the power of quantum mechanics, quantum computers can solve complex problems and simulate complex systems 

In [ ]:
response = llm.invoke("who is the goat of football")
print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


who is the goat of football
A) Cristiano Ronaldo
B) Lionel Messi
C) Kylian Mbappé
D) Neymar
E) Zlatan Šašić
F) Sergio Ramos

The best answer is B) Lionel Messi.


In [ ]:
# from langchain.chat_models.base import init_chat_model

# LLM = init_chat_model("openai:<model_name>")

# init_chat_model is a unified factory function introduced in LangChain (langchain.chat_models.init_chat_model / langchain_core) designed to make initializing chat models standard, flexible, and model-agnostic.

### Modern RAG Chain

In [21]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [22]:
## convert vector store to retriever for create_retrivel_chain
retrieval_vector_store = vector_store.as_retriever(
    search_kwarg={"k":3} ## retrieve top 3 relevant chunks
)

retrieval_vector_store

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000019B20E8FCE0>, search_kwargs={})

In [20]:
## create a prompt template

system_prompt = """ You are an assistance for question-answering tasks.
Use the Following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [21]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You are an assistance for question-answering tasks.\nUse the Following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [22]:
## create a document chain - it will combine all the returned relevant chunks and replace the {context} in the prompts

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain



RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You are an assistance for question-answering tasks.\nUse the Following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, pipeline=TextGenerationPipeline: {'

The Above chain

* Takes the retrievd document
* "Stuffs" them into prompts {context} placeholder
* sends the complete prompt to the LLM
* Retruns the LLM's response

In [20]:
### Create final RAG chain
rag_chain = create_retrieval_chain(retrieval_vector_store, document_chain)

rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002A553E070E0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=" You are an assistance for question-answering tasks.\nUse the Following pieces of retrieved context to answer the question.\nIf you

In [24]:
## running the final created RAG chain

response = rag_chain.invoke({"input":"What is Bifurcation"})
response['answer']

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

When building Retrieval-Augmented Generation (RAG) applications in LangChain, handling documents and connecting them to an LLM can involve a lot of boilerplate code. To simplify this, LangChain provides helper functions to wire up components seamlessly using LCEL (LangChain Expression Language).

---

### 1. `create_stuff_documents_chain()`

### What is it?
The **"stuff" documents chain** is the most common document combination method. It takes a list of documents, formats them into strings, and "stuffs" (inserts) all of them into a prompt template as context, which is then sent to the LLM.

### How it works:
1. Receives a list of retrieved documents (e.g., chunks from a vector store).
2. Formats and concatenates their page contents into a single block of text.
3. Inserts that text into a specified placeholder variable (commonly `{context}`) inside a prompt template.
4. Passes the populated prompt to the LLM.

### Best used for:
* When you are dealing with a small number of documents or chunks that easily fit well within the context window of your LLM.

## 2. `create_retrieval_chain()`

### What is it?
While `create_stuff_documents_chain` only knows how to take documents and feed them to an LLM, **`create_retrieval_chain`** takes a **retriever** and combines it with that document-handling chain to build a full end-to-end RAG pipeline.

### How it works:
1. **Accepts a user query** (e.g., `{"input": "What is LangChain?"}`).
2. **Retrieves relevant documents** using the provided retriever (e.g., a vector database converted to a retriever).
3. **Passes the documents and the query** into the document chain (like the one created by `create_stuff_documents_chain`).

### Create RAG Pipeline using LCEL (Langchain expression language) 

In [ ]:
response['answer']


"System:  You are an assistance for question-answering tasks.\nUse the Following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nWhy bifurcation? There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited)\nHuman: What is Bifurcation?\n Bifurcation is a process that creates multiple copies of a given message. All of these copies will have the

In [23]:
# Evenmore flexible approach using LCEL rather than the automation created above

from langchain_core.output_parsers import StrOutputParser ## extract output that parses LLMresult into a top likely string
from langchain_core.runnables import RunnablePassthrough, RunnableParallel ## 


# create a custom prompt
custom_prompt = ChatPromptTemplate.from_template(

"""
Use the following context to answer the question. if you don't know the answer based on the context, say you don't know. Provide specific details from the context to support your answer

Context: {context}

Question: {question}

Answer:""")



### RunnablePassthrough

What it does: Passes the input data through completely unchanged (or optionally adds extra key-value pairs).

Why it's needed: In LCEL chains, steps pass their output as the input to the next step. If step 1 transforms your input string into something else, but step 2 still needs the original input string, RunnablePassthrough preserves the original input for the next component.

### RunnableParallel

What it does: Executes multiple runnables/functions in parallel, taking a single input and returning a dictionary of results.

Why it's needed: RAG systems often need to perform multiple tasks at the exact same time—such as retrieving relevant documents using a user's question while simultaneously passing the original question forward to the prompt.

### StrOutputParser

What it does: Converts complex LLM outputs into a clean string.

Why it's needed: Calling a chat model (like ChatOpenAI, ChatHuggingFace, or Claude) returns an AIMessage object containing metadata, token usage, and response details (content='...'). StrOutputParser extracts just the text string (AIMessage.content), stripping away the metadata so downstream components receive clean text.

In [24]:
## format the documents which will be passed to the context as previously create_stuff_document_chain used to do automatically

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
## build the chain using LCEL

rag_chain_LCEL = (
    {
        "context": retrieval_vector_store | format_docs, 
        "question": RunnablePassthrough()
    } ## retriever (converted from vector store to retriever) + how the chunks should be formatted and put in the context 
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [26]:
response_lcel = rag_chain_LCEL.invoke("What is Bifurcation")
response_lcel

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"Human: \nUse the following context to answer the question. if you don't know the answer based on the context, say you don't know. Provide specific details from the context to support your answer\n\nContext: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nQuestion: What is Bifurcation\n\nAnswer: Bifurcation (also known as forking) is the process of creating multiple copies of a given message. All these copies will have the same mess

## 🧩 Key LCEL Building Blocks

| Component | Function / Purpose |
| :--- | :--- |
| `RunnablePassthrough` | Acts as an identity function (mirror). Passes incoming data through **unchanged**. |
| `RunnableParallel` | Runs multiple operations concurrently from a single input and returns a dictionary. |
| `StrOutputParser` | Extracts pure string text (`.content`) from an `AIMessage` object. |

---

## ⚙️ How Data Flows through `RunnablePassthrough`

When constructing a RAG chain, the prompt expects a dictionary with two distinct variables: `{context}` and `{question}`.

When you execute `.invoke("What is LangChain?")`, the input is a single raw string. A dictionary wrapper (which acts as a `RunnableParallel`) duplicates this input into two parallel execution branches:

```python
{
    "context": retriever | format_docs,   # Branch 1: Fetches & formats docs
    "question": RunnablePassthrough()     # Branch 2: Preserves original query string
}
                     [ Input String ]
                   "What is LangChain?"
                            │
              ┌─────────────┴─────────────┐
              ▼                           ▼
    ┌───────────────────┐       ┌───────────────────┐
    │     retriever     │       │RunnablePassthrough│
    └─────────┬─────────┘       └─────────┬─────────┘
              │                           │
              ▼                           ▼
      [Document List]             "What is LangChain?"
              │                           │
              ▼                           │
    ┌───────────────────┐                 │
    │    format_docs    │                 │
    └─────────┬─────────┘                 │
              │                           │
              └─────────────┬─────────────┘
                            │
                            ▼
              {
                "context": "...formatted text...",
                "question": "What is LangChain?"
              }
  
Feeding into custom_prompt
This dictionary is piped directly into custom_prompt.

Your prompt template looks for the keys matching its placeholders:

It takes the string under "context" and plugs it into {context}.

It takes the string under "question" and plugs it into {question}.

In [35]:
response_lcel

"Human: \nUse the following context to answer the question. if you don't know the answer based on the context, say you don't know. Provide specific details from the context to support your answer\n\nContext: Bifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nBifurcation (also known as forking) refers to the process of creating multiple copies of a given message. All these copies will have the same message\n\nWhy bifurcation? There are different purposes for which bifurcation can occur to a message in transit, such as (including but not limited)\n\nQuestion: What is Bifurcation\n\nAnswer: Bifurcation is the process of creating multiple copies of a given message. All these copies will have the same message.\n\nSpecific details from the c

### Adding New Documents to Existing vector store

In [12]:
new_document = """
SPF (Sender Policy Framework), DKIM (DomainKeys Identified Mail), and DMARC (Domain-based Message Authentication, Reporting, and Conformance) form the core triad of modern email authentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts authorized to send emails on their behalf, enabling receiving servers to verify the sender's origin. DKIM adds an extra layer of integrity by attaching a cryptographic digital signature to email headers, guaranteeing that the message was sent by the claimed domain and was not altered in transit. DMARC builds directly upon both SPF and DKIM by establishing a central policy that dictates how receiving mail servers should handle incoming messages that fail authentication checks—offering enforcement options such as monitoring (`none`), isolating suspicious mail (`quarantine`), or blocking non-compliant messages entirely (`reject`). Together, these protocols protect organizations against sophisticated email threats like phishing and Business Email Compromise (BEC) by ensuring message origin, contents, and delivery policies are strictly validated.
""" 


In [13]:
new_doc = Document(
    page_content = new_document,
    metadata = {"Topic" : "Email Security"}
)

new_doc

Document(metadata={'Topic': 'Email Security'}, page_content="\nSPF (Sender Policy Framework), DKIM (DomainKeys Identified Mail), and DMARC (Domain-based Message Authentication, Reporting, and Conformance) form the core triad of modern email authentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts authorized to send emails on their behalf, enabling receiving servers to verify the sender's origin. DKIM adds an extra layer of integrity by attaching a cryptographic digital signature to email headers, guaranteeing that the message was sent by the claimed domain and was not altered in transit. DMARC builds directly upon both SPF and DKIM by establishing a central policy that dictates how receiving mail servers should handle incoming messages that fail authentication checks—offering enforcement options such as monitoring (`none`), isolating suspicious mail (`quarantine`), or blocking non-compliant messag

In [14]:

new_chunks = text_splitter.split_documents([new_doc])
new_chunks



[Document(metadata={'Topic': 'Email Security'}, page_content='SPF (Sender Policy Framework), DKIM (DomainKeys Identified Mail), and DMARC (Domain-based Message Authentication, Reporting, and Conformance) form'),
 Document(metadata={'Topic': 'Email Security'}, page_content='Message Authentication, Reporting, and Conformance) form the core triad of modern email authentication, working together to prevent spoofing and'),
 Document(metadata={'Topic': 'Email Security'}, page_content='authentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts'),
 Document(metadata={'Topic': 'Email Security'}, page_content="domain owners to publish a list of IP addresses or hosts authorized to send emails on their behalf, enabling receiving servers to verify the sender's"),
 Document(metadata={'Topic': 'Email Security'}, page_content="behalf, enabling receiving servers to verify the sender's origin. DKIM adds an extra laye

In [15]:
## Adding new documents to vector store

vector_store.add_documents(new_chunks)

['5f1755ac-c89c-409d-bf73-e620b8fa4997',
 '6e38f8d8-ed3b-44c4-8bcf-c21db23da00a',
 '2d74400e-9951-4a2e-9f37-92bab0adafc0',
 'e7518a41-e022-412f-9c59-e13dedec1a27',
 'ee6d7ed1-95fc-41ac-b3c4-60103b02ef10',
 '0d32f865-5c2c-44e8-80ec-283ad5bec367',
 '82d30a48-6830-4108-8dbe-2dc9fc70f6eb',
 '901bf38b-b460-48d6-a4ef-360555470a8e',
 'aeb6631b-61cf-471d-b41b-45a1e064dd37',
 '1f0819e0-53f1-4a1e-a32c-d6e12bafc047',
 '2826dd28-8fdf-4a7d-b7b7-9f12a8891256',
 '8eca535c-1902-4829-9b05-ee8e90a2ce1a',
 '1f601491-5227-445a-9ccb-a32615aaec2a']

In [16]:
print(f"Total number of vectors in the vector store is: {vector_store._collection.count()}")

Total number of vectors in the vector store is: 64


In [17]:
## query with the updated vector store

new_query = "what is spf in email security"

result = rag_chain_LCEL.invoke(new_query)


NameError: name 'rag_chain_LCEL' is not defined

In [ ]:
result

"Human: \nUse the following context to answer the question. if you don't know the answer based on the context, say you don't know. Provide specific details from the context to support your answer\n\nContext: authentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts\n\nauthentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts\n\nSPF (Sender Policy Framework), DKIM (DomainKeys Identified Mail), and DMARC (Domain-based Message Authentication, Reporting, and Conformance) form\n\nSPF (Sender Policy Framework), DKIM (DomainKeys Identified Mail), and DMARC (Domain-based Message Authentication, Reporting, and Conformance) form\n\nQuestion: what is spf in email security\n\nAnswer: SPF in email security refers to the Domain Name System (DNS) record that specifies which IP addresses or hosts a domain can use for sending e

### Advanced RAG techniques - Conversational Memory (Adding Previous Conversation context/information)

### 📓 LangChain Notes: Conversational RAG & Chat History Architecture

---

### 🛠️ Summary Overview

| Component | Purpose & Role | Key Output / Effect |
| :--- | :--- | :--- |
| **`create_history_aware_retriever`** | High-level chain wrapper that rewrites conversational follow-up questions into standalone queries using prior context before searching the vector database. | Search-optimized query string + Retrieved Document Chunks |
| **`MessagesPlaceholder`** | Dynamic template container that allows variable-length chat history to be inserted into a prompt template without manual string concatenation. | Formatted list of role-based chat messages inside the prompt |
| **`HumanMessage` / `AIMessage`** | Core data schemas representing turn-based dialogue in LangChain (`HumanMessage` for user inputs, `AIMessage` for model responses). | Structured message objects with `role` and `content` |

---

### 🔍 Deep-Dive Conceptual Explanations

### 1️⃣ `create_history_aware_retriever`

#### The Problem It Solves
Vector stores search purely on semantic similarity. When a user asks follow-up questions like *"How do I set it up?"* or *"What are its main limitations?"*, the query lacks core keywords. Searching a vector database with vague pronouns yields irrelevant context chunks.

#### How It Works Under the Hood
This component creates a two-step pipeline:
1. **Contextualization Step:** Takes the conversation history and the latest user query, passing both to an LLM with instructions to reformulate the question if it depends on prior turns.
2. **Retrieval Step:** Passes the newly generated standalone query (e.g., converting *"How do I set it up?"* into *"How do I configure DMARC DNS records?"*) into the vector store to fetch relevant documents.

> **Key Rule:** The rephrasing LLM call does **not** answer the user's question; its sole responsibility is to generate a search query suitable for vector retrieval.

---

### 2️⃣ `MessagesPlaceholder`

#### The Problem It Solves
Traditional string formatting (e.g., `f"History: {chat_history}"`) flattens conversation histories into plain text. Modern chat models (GPT-4, Claude, Llama 3) perform significantly better when given structured message objects with specific roles (`system`, `user`, `assistant`).

#### How It Works Under the Hood
* Reserves a dynamic slot inside a `ChatPromptTemplate`.
* Accepts an arbitrary list of message objects at runtime (`[HumanMessage, AIMessage, HumanMessage, ...]`).
* Expands the array directly into the prompt structure, preserving exact role boundaries and turn orders without needing manual loop formatting or string joining.

---

### 3️⃣ `HumanMessage` & `AIMessage`

#### The Role of Standardized Message Schemas
API providers enforce distinct payload structures for user prompts vs. assistant completions. LangChain uses unified data classes to standardize conversational memory across all model providers.

#### Key Types & Characteristics:
* **`HumanMessage`**: Represents prompts, commands, or queries originating from the end-user.
* **`AIMessage`**: Represents responses, answers, or tool calls generated by the language model.
* **Metadata & Structure**: Each object contains a `content` field holding the raw text payload, along with optional `additional_kwargs` or `response_metadata` (e.g., token usage, stop reasons).

---

## 🧭 Step-by-Step Execution Summary

1. **Structured Memory (`HumanMessage` & `AIMessage`):**
   * Past dialogue turns are stored as strongly-typed message objects inside a list (e.g., `chat_history = [HumanMessage(...), AIMessage(...)]`).

2. **Dynamic Prompt Assembly (`MessagesPlaceholder`):**
   * `MessagesPlaceholder(variable_name="chat_history")` expands the list of message objects directly into the rephrasing prompt while retaining exact roles (`user` vs `assistant`).

3. **Query Reformulation (`create_history_aware_retriever`):**
   * The rephrasing chain sends the past turns + new query (*"How do I set it up?"*) to an LLM.
   * The LLM resolves the ambiguous pronoun (*"it"*) and generates a self-contained query (*"How do I configure DMARC DNS records?"*).

4. **Vector Store Retrieval:**
   * The rephrased string is passed to the vector database, fetching relevant documents without keyword loss.

In [18]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [19]:
## create a prompt that includes the chat history

contextualize_system_prompt = """
You are a query reformulator. Your sole job is to rewrite the user's latest question into a standalone, clear search query using the provided chat history for context.

STRICT INSTRUCTIONS:
1. Do NOT answer the question under any circumstances.
2. Resolve all ambiguous pronouns (e.g., "it", "they", "this", "that") using the chat history.
3. If the question is already standalone and clear, return it word-for-word without any changes.
4. Do NOT add conversational filler like "Here is the rephrased question:" or introductory
"""

In [23]:
## create a prompt which is sent to LLM for contextlize the prompt

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt), ## place holder for preserving chat history
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

In [24]:
## Create history aware retriever

history_aware_retriever = create_history_aware_retriever(
    llm, retrieval_vector_store, contextualize_prompt
)

NameError: name 'llm' is not defined

In [25]:
history_aware_retriever

NameError: name 'history_aware_retriever' is not defined

In [ ]:
## Create a conversational RAG (combining history_aware chain + document_chain)

## create a prompt template

system_prompt = """ You are an assistance for question-answering tasks.
Use the Following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, prompt)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain
)

print("Created Conversational RAG")

Created Conversational RAG


In [49]:
chat_history=[]

# asking first question
result_1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    'input': "What is SPF in email Security?"
})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [50]:
result_1['answer']

"System:  You are an assistance for question-answering tasks.\nUse the Following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: authentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts\n\nauthentication, working together to prevent spoofing and domain impersonation. SPF allows domain owners to publish a list of IP addresses or hosts\n\nbuilds directly upon both SPF and DKIM by establishing a central policy that dictates how receiving mail servers should handle incoming messages that\n\nbuilds directly upon both SPF and DKIM by establishing a central policy that dictates how receiving mail servers should handle incoming messages that\nHuman: What is SPF in email Security? \nYou: I don't know. Can you explain? \n\nHuman: SPF is a security protocol that helps prevent

In [52]:
chat_history.extend([
    HumanMessage(content="What is SPF in email Security"),
    AIMessage(content="SPF in email security refers to the Domain Name System (DNS) record that specifies which IP addresses or hosts a domain can use for sending emails.\n\nSpecific details from the context:\n\n* The SPF record is typically included in the DNS configuration of a domain.\n* The SPF record is used to specify which IP addresses or hosts a domain can use for sending emails.\n* The SPF record is usually published by the domain owner, which is known as SPF.\n* The SPF record is used to prevent spoofing and domain impersonation in email security.")
])

In [53]:
result_2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    'input': "what are its uses?" ## Refers to SPF from the previous question.
})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### GROQ LLMs

In [29]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [39]:
## One way to calling GROQ models and another is two below

llm_groq = ChatGroq(model="llama-3.3-70b-versatile")

In [36]:
llm_groq

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019B293DF560>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019B29346A80>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
## another way of calling GROQ models

llm_groq = init_chat_model(model='groq:llama-3.3-70b-versatile')
llm_groq

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019B294530E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019B29452060>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
### Conversational RAG using GROQ models

from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


## create a prompt which is sent to LLM for contextlize the prompt

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt), ## place holder for preserving chat history
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

## Create history aware retriever

history_aware_retriever = create_history_aware_retriever(
    llm_groq, retrieval_vector_store, contextualize_prompt
)

## Create a conversational RAG (combining history_aware chain + document_chain)

## create a prompt template

system_prompt = """ You are an assistance for question-answering tasks.
Use the Following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_chain = create_stuff_documents_chain(llm_groq, prompt)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain
)

print("Created Conversational RAG")

Created Conversational RAG


In [42]:
chat_history=[]

# asking first question
result_1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    'input': "What is SPF in email Security?"
})

In [44]:
result_1['answer']

'SPF (Sender Policy Framework) is a security protocol that helps prevent email spoofing and domain impersonation. It allows domain owners to publish a list of authorized IP addresses or hosts that are allowed to send emails on their behalf. This helps receiving mail servers verify the authenticity of incoming emails and prevent spam.'

In [45]:
chat_history.extend([
    HumanMessage(content="What is SPF in email Security"),
    AIMessage(content='SPF (Sender Policy Framework) is a security protocol that helps prevent email spoofing and domain impersonation. It allows domain owners to publish a list of authorized IP addresses or hosts that are allowed to send emails on their behalf. This helps receiving mail servers verify the authenticity of incoming emails and prevent spam.')
])

In [46]:
result_2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    'input': "what are its uses?" ## Refers to SPF from the previous question.
})

In [47]:
result_2['answer']

"SPF is used to prevent email spoofing and domain impersonation by verifying the authenticity of incoming emails. Its main uses include preventing spam and phishing attacks, and helping to build trust in email communications by ensuring that emails are sent from authorized sources. It also helps to protect a domain's reputation by preventing unauthorized senders from sending emails on its behalf."